# ExplainPlan-Vision
## Phase 2: Explainability Engine

**Project:** ExplainPlan-Vision — Explainable Neuro-Symbolic Visual Planning Agent  
**Phase:** 2 of 7 — Explainability Engine  
**Depends on:** Phase 1 outputs (best_model.pth, class_mappings.json, config.json)  
**Author:** Muhammad Aqib Javed  
**Date:** 2026

### What this phase builds

Phase 1 gave us a vision model that classifies plant diseases with near-perfect accuracy. Accuracy alone is not sufficient for a trustworthy agricultural AI system. A farmer who receives a treatment recommendation needs to understand *why* the system made that decision before acting on it. This phase builds the explainability infrastructure that makes that trust possible.

We implement three complementary explanation methods, each answering a different question about the model's behaviour:

| Method | Question answered | Type |
|--------|------------------|------|
| Grad-CAM | Which spatial regions of the leaf image drove the prediction? | Visual / spatial |
| SHAP | Which dimensions of the learned feature embedding contributed most? | Feature attribution |
| LIME | What local input perturbations would flip the prediction? | Decision boundary |

Together they form a three-layer explanation: where the model looked, what internal features mattered, and how fragile the decision boundary is. This multi-method approach is what separates an explainability *engine* from a simple heatmap overlay.

### How Phase 2 connects to later phases

```
Phase 1 output
    {disease, confidence, severity, feature_embedding}
    ↓
Phase 2 adds
    {gradcam_heatmap, shap_attribution, lime_mask,
     attention_summary, why_not_explanation, counterfactual}
    ↓
Phase 3 Planning Engine consumes the full enriched output
    to generate treatment step sequences
    ↓
Phase 5 Human-Aware Explanation selects which explanation
    to show based on the user's expertise level
```

### Phase 2 objectives

| Objective | Status |
|-----------|--------|
| Load Phase 1 model without retraining | planned |
| Grad-CAM on EfficientNet last conv layer | planned |
| SHAP DeepExplainer on feature embeddings | planned |
| LIME superpixel explanation | planned |
| Why-not explanation (top-2 contrastive) | planned |
| Counterfactual via embedding nearest-neighbour | planned |
| Unified explain() function | planned |
| Research comparison figure (all three methods) | planned |
| Export explainability outputs for Phase 3 | planned |

---
## How to set up this notebook on Kaggle

**Step 1 — Save Phase 1 outputs as a Kaggle dataset**

After Phase 1 finishes, go to your Phase 1 notebook on Kaggle. In the right sidebar, find the Output section. Click the folder icon to browse `outputs/`. Select the entire `outputs/` directory and click "New Dataset". Name it something like `explainplan-phase1-outputs`. Kaggle will package everything including `best_model.pth`, `class_mappings.json`, `config.json`, and the log files.

**Step 2 — Add that dataset as input to this notebook**

In this Phase 2 notebook, click "Add data" in the right sidebar. Search for your dataset by name. Once added, the files will be available at `/kaggle/input/explainplan-phase1-outputs/`. Update the `PHASE1_DIR` variable in Cell 3 to match.

**Step 3 — Add the PlantVillage dataset again**

Phase 2 needs a small sample of real images for LIME and the comparison figures. Add the same PlantVillage dataset you used in Phase 1.

**Why this uses almost no GPU time**

We load the frozen Phase 1 model and run forward passes on a small number of images. No training occurs. Grad-CAM, SHAP, and LIME are inference-time operations. The entire notebook should run in under 30 minutes of GPU time.

---
## Cell 1 — Install dependencies

We install the three XAI libraries on top of the Phase 1 stack. `grad-cam` is the pytorch-grad-cam library by Jacob Gildenblat, which supports EfficientNet natively. `shap` provides the DeepExplainer. `lime` provides the image explainer.

In [ ]:
!pip install -q albumentations timm grad-cam shap lime

---
## Cell 2 — Imports

In [ ]:
# Standard library
import os
import json
import random
import warnings
from pathlib import Path

# Numerical
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Image processing
import cv2
from PIL import Image

# Progress
from tqdm.auto import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

# Augmentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Model zoo
import timm

# XAI libraries
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

# Metrics
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

---
## Cell 3 — Phase 2 configuration

Most parameters are loaded directly from Phase 1's saved `config.json`. We only add Phase 2 specific settings here. This ensures that both phases always use identical image size, normalization statistics, and model architecture — a consistency requirement for reproducibility.

Update `PHASE1_DIR` to match where you saved your Phase 1 outputs dataset.

In [ ]:
# Path to your Phase 1 outputs dataset on Kaggle.
# After saving Phase 1 outputs as a Kaggle dataset and adding it as input,
# the files appear at /kaggle/input/<your-dataset-name>/
PHASE1_DIR = "/kaggle/input/datasets/maqibniazi/explainplan-phase1-outputs"

# PlantVillage dataset — same as Phase 1
DATASET_PATH = "/kaggle/input/datasets/emmarex/plantdisease/PlantVillage"

# Load Phase 1 config — this is the single source of truth for all
# image preprocessing parameters so the two phases stay in sync.
with open(f"{PHASE1_DIR}/outputs/checkpoints/config.json") as f:
    P1_CONFIG = json.load(f)

# Load class mappings
with open(f"{PHASE1_DIR}/outputs/checkpoints/class_mappings.json") as f:
    mappings = json.load(f)

class_to_idx = mappings["class_to_idx"]
idx_to_class = {int(k): v for k, v in mappings["idx_to_class"].items()}
classes      = mappings["classes"]
NUM_CLASSES  = mappings["num_classes"]

# Phase 2 specific settings
CONFIG = {
    # Inherited from Phase 1 — do not change these manually
    "model_name"    : P1_CONFIG["model_name"],
    "image_size"    : P1_CONFIG["image_size"],
    "mean"          : P1_CONFIG["mean"],
    "std"           : P1_CONFIG["std"],
    "embedding_dim" : P1_CONFIG["embedding_dim"],
    "dropout"       : P1_CONFIG["dropout"],
    "device"        : "cuda" if torch.cuda.is_available() else "cpu",

    # Phase 2 specific
    "checkpoint_path"  : f"{PHASE1_DIR}/outputs/checkpoints/best_model.pth",
    "xai_output_dir"   : "outputs/xai",
    "n_demo_images"    : 8,       # images for the comparison figure
    "lime_num_samples" : 1000,    # perturbation samples for LIME
    "lime_num_features": 10,      # superpixel segments to highlight
    "shap_background"  : 50,      # background samples for SHAP DeepExplainer
    "seed"             : 42,
}

# Create output directories
for subdir in ["gradcam", "shap", "lime", "comparison", "counterfactual"]:
    os.makedirs(f"{CONFIG['xai_output_dir']}/{subdir}", exist_ok=True)

print(f"Phase 1 config loaded")
print(f"  Model        : {CONFIG['model_name']}")
print(f"  Image size   : {CONFIG['image_size']}")
print(f"  Classes      : {NUM_CLASSES}")
print(f"  Checkpoint   : {CONFIG['checkpoint_path']}")

---
## Cell 4 — Reproducibility

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ["PYTHONHASHSEED"]       = str(seed)

set_seed(CONFIG["seed"])
print(f"Seed set to {CONFIG['seed']}")

---
## Cell 5 — Preprocessing pipeline

We redeclare the inference transform from Phase 1 using the loaded normalization constants. This cell is the only preprocessing definition in Phase 2; nothing else should resize or normalize images.

In [ ]:
inference_transform = A.Compose([
    A.Resize(CONFIG["image_size"], CONFIG["image_size"]),
    A.Normalize(mean=CONFIG["mean"], std=CONFIG["std"]),
    ToTensorV2()
])

def load_image_rgb(path):
    """Load an image from disk as a uint8 RGB numpy array."""
    img = cv2.imread(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def preprocess_for_model(image_rgb):
    """
    Apply the inference transform and return a (1, 3, H, W) tensor
    on the configured device.
    """
    tensor = inference_transform(image=image_rgb)["image"]
    return tensor.unsqueeze(0).to(CONFIG["device"])

def preprocess_for_display(image_rgb):
    """
    Resize to model input size but do NOT normalize.
    Used as the RGB float32 base for Grad-CAM overlays.
    """
    resized = cv2.resize(image_rgb, (CONFIG["image_size"], CONFIG["image_size"]))
    return resized.astype(np.float32) / 255.0

print("Preprocessing pipeline ready")

---
## Cell 6 — Model definition

This is an exact copy of the `PlantDiseaseModel` class from Phase 1. We redeclare it here so the notebook is self-contained. The architecture must be identical to Phase 1 for the weights to load correctly — do not change anything in this class definition.

In [ ]:
class PlantDiseaseModel(nn.Module):
    """
    Modular perception backbone for ExplainPlan-Vision.

    This definition must remain identical to Phase 1 so that the
    saved checkpoint loads without key mismatches.
    """

    def __init__(self, model_name, num_classes, embedding_dim=512, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0)
        backbone_out  = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(embedding_dim, num_classes)
        )
        self.embedding_dim = embedding_dim

    def forward(self, x):
        features = self.backbone(x)
        logits   = self.classifier(features)
        return logits, features

    def extract_features(self, x):
        with torch.no_grad():
            return self.backbone(x)


model = PlantDiseaseModel(
    model_name    = CONFIG["model_name"],
    num_classes   = NUM_CLASSES,
    embedding_dim = CONFIG["embedding_dim"],
    dropout       = CONFIG["dropout"]
).to(CONFIG["device"])

model.load_state_dict(
    torch.load(CONFIG["checkpoint_path"],
               map_location=CONFIG["device"],
               weights_only=True)
)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {CONFIG['model_name']}")
print(f"Parameters  : {total_params:,}")
print(f"Mode        : eval (frozen — no training in Phase 2)")

---
## Cell 7 — Sample image loader

We build a small representative image bank — two images per disease class — used throughout Phase 2 for all explanation demonstrations. Using the same set for all three XAI methods is important: it lets us produce a direct side-by-side comparison figure later.

In [ ]:
dataset_path = Path(DATASET_PATH)
set_seed(CONFIG["seed"])

# Collect two random images per class
sample_bank = []  # list of {path, label, plant, disease}

for cls in classes:
    cls_dir = dataset_path / cls
    images  = [f for f in cls_dir.iterdir() if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    chosen  = random.sample(images, min(2, len(images)))

    for img_path in chosen:
        if "___" in cls:
            parts   = cls.split("___", 1)
            plant   = parts[0].replace("_", " ").strip()
            disease = parts[1].replace("_", " ").strip()
        else:
            tokens  = cls.split("_")
            plant   = tokens[0].strip()
            disease = " ".join(tokens[1:]).strip()

        sample_bank.append({
            "path"    : str(img_path),
            "label"   : cls,
            "plant"   : plant,
            "disease" : disease,
        })

# Separate demo images (one per class, up to n_demo_images)
demo_images = [sample_bank[i * 2] for i in range(min(CONFIG["n_demo_images"], NUM_CLASSES))]

print(f"Sample bank  : {len(sample_bank)} images across {NUM_CLASSES} classes")
print(f"Demo images  : {len(demo_images)}")

---
## Cell 8 — Base inference function

This is the same `predict()` function from Phase 1, with one addition: it now returns the raw logits tensor as well as the softmax probabilities. The logits are needed by SHAP.

In [ ]:
def estimate_severity(confidence, config):
    t = config.get("severity_thresholds",
                   {"high": 0.85, "medium": 0.60, "low": 0.0})
    if confidence >= t["high"]:
        return "high"
    elif confidence >= t["medium"]:
        return "medium"
    return "low"


def predict(image_rgb, model, config):
    """
    Run inference on a uint8 RGB numpy image.

    Returns a dictionary containing the prediction, confidence,
    severity, top-3 predictions, and the raw feature embedding.
    Also returns the logits tensor (on CPU) for downstream XAI use.
    """
    model.eval()
    tensor = preprocess_for_model(image_rgb)

    with torch.no_grad():
        with autocast("cuda", enabled=torch.cuda.is_available()):
            logits, features = model(tensor)
        probs = torch.softmax(logits.float(), dim=1)[0]

    top_idx    = probs.argmax().item()
    confidence = probs[top_idx].item()
    label_name = idx_to_class[top_idx]

    if "___" in label_name:
        parts        = label_name.split("___", 1)
        plant        = parts[0].replace("_", " ").strip()
        disease_type = parts[1].replace("_", " ").strip()
    else:
        tokens       = label_name.split("_")
        plant        = tokens[0].strip()
        disease_type = " ".join(tokens[1:]).strip()

    top3_vals, top3_idxs = torch.topk(probs, k=min(3, NUM_CLASSES))
    top3 = [
        {"class": idx_to_class[i.item()], "confidence": round(v.item(), 4)}
        for v, i in zip(top3_vals, top3_idxs)
    ]

    return {
        "disease"          : label_name,
        "plant"            : plant,
        "disease_type"     : disease_type,
        "confidence"       : round(confidence, 4),
        "severity"         : estimate_severity(confidence, config),
        "top3_predictions" : top3,
        "feature_embedding": features[0].cpu().float().numpy(),
        "is_healthy"       : "healthy" in label_name.lower(),
        "logits_cpu"       : logits.cpu().float()
    }


print("predict() function ready")

---
## Cell 9 — Grad-CAM engine

### Background

Gradient-weighted Class Activation Mapping (Grad-CAM) computes the gradient of the class score with respect to the final convolutional feature map. Regions where the gradient is large are the regions that most influenced the prediction. For EfficientNet-B0, the target layer is `model.backbone.conv_head` — the last convolutional layer before global average pooling.

We use `GradCAMPlusPlus` rather than vanilla Grad-CAM because it produces sharper, more localised activations on small lesion regions, which is precisely the use case here. The improvement comes from computing second-order gradients, which weights the contributions more fairly when the activation map contains multiple small regions.

### Why this layer?

EfficientNet's `conv_head` is the last layer before the global pooling operation collapses spatial information. It retains a 7x7 spatial grid (for 224x224 input) while already encoding high-level semantic features. Attaching Grad-CAM here gives the best trade-off between spatial resolution and semantic content.

In [ ]:
class GradCAMWrapper(nn.Module):
    """
    Adapter that makes PlantDiseaseModel compatible with pytorch-grad-cam.

    The library expects model(x) to return a plain (batch, num_classes) tensor.
    Our model returns (logits, features), so this wrapper extracts only logits.
    The original model is not modified and continues to expose embeddings for
    SHAP, LIME, and the Phase 3 planning engine.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        logits, _ = self.model(x)
        return logits


class GradCAMEngine:
    """
    Grad-CAM++ wrapper for EfficientNet-B0 loaded via timm.

    Target layer is backbone.conv_head — the last convolutional layer before
    global average pooling. At 224x224 input this produces a 7x7 activation
    map, giving a good balance between spatial resolution and semantic content.
    """

    def __init__(self, model, device):
        self.model         = model
        self.device        = device
        self.wrapped_model = GradCAMWrapper(model).to(device)
        self.wrapped_model.eval()

        # The target layer path goes through the wrapper
        target_layer = [self.wrapped_model.model.backbone.conv_head]

        self.cam = GradCAMPlusPlus(
            model=self.wrapped_model,
            target_layers=target_layer
        )

    def generate(self, image_rgb, target_class_idx=None):
        """
        Generate a Grad-CAM++ heatmap and overlay for the given image.

        Parameters
        ----------
        image_rgb        : uint8 numpy array (H, W, 3)
        target_class_idx : int or None. If None, uses the predicted class.

        Returns
        -------
        heatmap  : float32 numpy array in [0,1], shape (H, W)
        overlay  : uint8 numpy array (H, W, 3), heatmap blended onto image
        """
        tensor    = preprocess_for_model(image_rgb)
        rgb_float = preprocess_for_display(image_rgb)

        targets = [ClassifierOutputTarget(target_class_idx)] if target_class_idx is not None else None

        heatmap = self.cam(input_tensor=tensor, targets=targets)
        heatmap = heatmap[0]  # (H, W)

        overlay = show_cam_on_image(rgb_float, heatmap, use_rgb=True)
        return heatmap, overlay

    def generate_counterfactual(self, image_rgb, runner_up_class_idx):
        """
        Grad-CAM for the runner-up class.
        Answers: what would the model focus on if it believed this was class X?
        """
        return self.generate(image_rgb, target_class_idx=runner_up_class_idx)


gradcam_engine = GradCAMEngine(model, CONFIG["device"])
print("GradCAMEngine ready")
print("  Target layer : wrapped_model.model.backbone.conv_head")

---
## Cell 10 — Grad-CAM demo on representative images

We generate heatmaps for all demo images and save them. The output for each image is:
- The original resized image
- The Grad-CAM heatmap (grayscale)
- The overlay (heatmap blended onto image)
- The counterfactual overlay (heatmap for the runner-up class)

In [ ]:
gradcam_results = []

for sample in tqdm(demo_images, desc="Grad-CAM"):
    image_rgb  = load_image_rgb(sample["path"])
    prediction = predict(image_rgb, model, CONFIG)

    heatmap, overlay = gradcam_engine.generate(image_rgb)

    # Counterfactual: Grad-CAM for the runner-up class
    runner_up_idx   = class_to_idx[prediction["top3_predictions"][1]["class"]]
    _, cf_overlay   = gradcam_engine.generate_counterfactual(image_rgb, runner_up_idx)

    # Save outputs
    label_safe = sample["label"].replace("/", "_")
    cv2.imwrite(
        f"{CONFIG['xai_output_dir']}/gradcam/{label_safe}_overlay.png",
        cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    )

    gradcam_results.append({
        "sample"      : sample,
        "prediction"  : prediction,
        "image_rgb"   : image_rgb,
        "heatmap"     : heatmap,
        "overlay"     : overlay,
        "cf_overlay"  : cf_overlay,
        "runner_up"   : prediction["top3_predictions"][1]["class"],
    })

print(f"Grad-CAM generated for {len(gradcam_results)} images")

---
## Cell 11 — Grad-CAM visualisation grid

This figure is designed for direct inclusion in a research paper. Each row shows one disease class. The columns are: original image, Grad-CAM overlay (predicted class), and Grad-CAM overlay for the runner-up class (counterfactual). The counterfactual column shows where the model would have looked if it had predicted the second-most-likely disease, which grounds the why-not explanation.

In [ ]:
n  = len(gradcam_results)
fig, axes = plt.subplots(n, 3, figsize=(13, n * 3.2))

col_titles = ["Original", "Grad-CAM++ (predicted)", "Grad-CAM++ (runner-up)"]
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=10, fontweight="bold", pad=8)

for i, res in enumerate(gradcam_results):
    original  = cv2.resize(res["image_rgb"], (224, 224))
    pred_name = res["prediction"]["disease_type"]
    conf      = res["prediction"]["confidence"]
    runner_up = res["runner_up"].split("___")[-1].replace("_", " ") if "___" in res["runner_up"] \
                else " ".join(res["runner_up"].split("_")[1:])

    axes[i, 0].imshow(original)
    axes[i, 0].set_ylabel(
        f"{res['sample']['plant']}\n{pred_name}",
        fontsize=7, rotation=0, labelpad=60, va="center"
    )

    axes[i, 1].imshow(res["overlay"])
    axes[i, 1].set_xlabel(f"conf={conf:.3f}", fontsize=7)

    axes[i, 2].imshow(res["cf_overlay"])
    axes[i, 2].set_xlabel(f"if: {runner_up[:25]}", fontsize=7)

    for j in range(3):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.suptitle(
    "Grad-CAM++ Spatial Explanations — ExplainPlan-Vision Phase 2",
    fontsize=12, fontweight="bold", y=1.005
)
plt.tight_layout()
plt.savefig(f"{CONFIG['xai_output_dir']}/gradcam/gradcam_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grad-CAM grid saved")

---
## Cell 12 — SHAP engine

### Background

SHAP (SHapley Additive exPlanations) is grounded in cooperative game theory. It assigns each input feature a Shapley value: the average marginal contribution of that feature across all possible feature coalitions. For deep neural networks, the `DeepExplainer` uses a linear approximation that makes it tractable.

### What we explain with SHAP

Rather than applying SHAP to raw pixels (which is very slow and produces noisy results for CNNs), we apply it to the 1280-dimensional backbone feature embedding. This tells us which of the learned visual concepts — encoded as individual embedding dimensions — drove the classification decision. This approach is faster, more stable, and produces cleaner attributions.

### Background sample selection

SHAP DeepExplainer requires a background dataset representing the "null" state — the expected output when no informative input is present. We use a random sample of 50 images from across all disease classes. A diverse background ensures that SHAP attributions capture deviations from the global average rather than class-specific baselines.

In [ ]:
def disable_inplace_activations(model):
    """
    Set inplace=False on all activation modules recursively.
    Required because SHAP's backward hooks conflict with SiLU(inplace=True)
    in EfficientNet, causing 'view is being modified inplace' RuntimeError.
    """
    for module in model.modules():
        if hasattr(module, "inplace"):
            module.inplace = False


class SHAPModelWrapper(nn.Module):
    """
    nn.Module wrapper so shap.GradientExplainer receives a proper module.
    Extracts only logits from the (logits, features) tuple our model returns.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        logits, _ = self.model(x)
        return logits


class SHAPEngine:
    """
    SHAP GradientExplainer for pixel-level attribution on EfficientNet.

    GradientExplainer is used instead of DeepExplainer because EfficientNet's
    Squeeze-and-Excitation blocks and depthwise separable convolutions violate
    the linear decomposition assumption that DeepExplainer requires. This
    shows up as a large additivity error (max diff ~5.0 vs tolerance 0.01).

    GradientExplainer works by sampling random convex combinations of the input
    and a set of background images, then computing gradients at each sample.
    It is compatible with arbitrary architectures and produces stable
    pixel-level attributions.
    """

    def __init__(self, model, background_paths, device, n_samples=50):
        self.device = device

        # Must disable inplace activations before any SHAP hooks are installed
        disable_inplace_activations(model)
        print("In-place activations disabled")

        self.shap_model = SHAPModelWrapper(model).to(device)
        self.shap_model.eval()

        # Build background tensor on the same device as the model.
        # GradientExplainer uses these as interpolation anchors, not a
        # distribution expectation, so 30-50 diverse images is sufficient.
        print(f"Building SHAP background from {n_samples} images...")
        bg_tensors = []
        chosen = random.sample(background_paths, min(n_samples, len(background_paths)))
        for path in tqdm(chosen, desc="Background", leave=False):
            img    = load_image_rgb(path)
            tensor = inference_transform(image=img)["image"]  # (3, H, W) CPU float
            bg_tensors.append(tensor.unsqueeze(0))

        # GradientExplainer keeps the background on the same device as the model
        background = torch.cat(bg_tensors, dim=0).float().to(device)
        print(f"Background tensor: {background.shape}")

        # GradientExplainer signature: (model, data)
        # data can be a tensor directly — no additivity constraint
        self.explainer  = shap.GradientExplainer(self.shap_model, background)
        self.n_bg       = background.shape[0]
        print("SHAP GradientExplainer ready")

    def explain(self, image_rgb, top_class_idx):
        """
        Compute SHAP gradient attribution for the predicted class.

        Returns
        -------
        class_shap   : numpy array (3, H, W) per-channel attribution
        shap_display : numpy array (H, W) mean absolute attribution
                       normalized to [0, 1]
        """
        H = W = CONFIG["image_size"]

        tensor = inference_transform(image=image_rgb)["image"]
        tensor = tensor.unsqueeze(0).float().to(self.device)

        shap_values, indices = self.explainer.shap_values(
            tensor,
            ranked_outputs=1,
            nsamples=50
        )

        # shap_values[0] shape varies across SHAP versions:
        # (1, 3, H, W), (1, 3, H, 1), (3, H, W), (3, H, 1), etc.
        # We normalize to (3, H, W) unconditionally.
        raw = np.array(shap_values[0])  # force numpy
        raw = raw.squeeze()             # remove all size-1 dims

        # After squeeze: could be (3, H, W), (H, W, 3), (H,), etc.
        # Handle each case explicitly.
        if raw.ndim == 3:
            if raw.shape[0] == 3:
                # Already (3, H, W)
                class_shap = raw
            elif raw.shape[2] == 3:
                # Channel-last (H, W, 3) — transpose to (3, H, W)
                class_shap = raw.transpose(2, 0, 1)
            else:
                # Unknown 3D layout — use mean across first axis
                class_shap = np.stack([raw] * 3, axis=0)
        elif raw.ndim == 2:
            # Single channel (H, W) — replicate to (3, H, W)
            class_shap = np.stack([raw] * 3, axis=0)
        elif raw.ndim == 1:
            # Fully collapsed — reshape to (3, H, W)
            total = raw.size
            if total == 3 * H * W:
                class_shap = raw.reshape(3, H, W)
            else:
                class_shap = np.stack([raw.reshape(H, W)] * 3, axis=0)
        else:
            raise ValueError(f"Cannot handle shap_values shape: {raw.shape}")

        # Ensure class_shap is exactly (3, H, W)
        if class_shap.shape != (3, H, W):
            class_shap = np.resize(class_shap, (3, H, W))

        shap_display = np.mean(np.abs(class_shap), axis=0)   # (H, W)
        assert shap_display.shape == (H, W), \
            f"shap_display still wrong: {shap_display.shape}"

        lo, hi = shap_display.min(), shap_display.max()
        shap_display = (shap_display - lo) / (hi - lo + 1e-8)

        return class_shap, shap_display

# Collect all image paths for background sampling
all_paths = []
for cls in classes:
    cls_dir = dataset_path / cls
    all_paths.extend([
        str(f) for f in cls_dir.iterdir()
        if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])

print(f"Background pool: {len(all_paths)} images across {NUM_CLASSES} classes")

shap_engine = SHAPEngine(
    model,
    all_paths,
    CONFIG["device"],
    n_samples=CONFIG["shap_background"]
)

---
## Cell 13 — SHAP explanations for demo images

In [ ]:
shap_results = []

for res in tqdm(gradcam_results, desc="SHAP"):
    image_rgb  = res["image_rgb"]
    prediction = res["prediction"]
    top_idx    = class_to_idx[prediction["disease"]]

    class_shap, shap_display = shap_engine.explain(image_rgb, top_idx)

    shap_results.append({
        "sample"       : res["sample"],
        "prediction"   : prediction,
        "image_rgb"    : image_rgb,
        "class_shap"   : class_shap,
        "shap_display" : shap_display,
    })

print(f"SHAP attributions computed for {len(shap_results)} images")

---
## Cell 14 — SHAP visualisation grid

In [ ]:
n   = len(shap_results)
fig, axes = plt.subplots(n, 3, figsize=(13, n * 3.2))

col_titles = ["Original", "SHAP attribution", "SHAP overlay"]
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=10, fontweight="bold", pad=8)

for i, res in enumerate(shap_results):
    original     = cv2.resize(res["image_rgb"], (224, 224))
    shap_display = res["shap_display"]
    conf         = res["prediction"]["confidence"]
    disease      = res["prediction"]["disease_type"]

    # SHAP overlay: apply colormap and blend
    shap_colored = cm.RdBu_r(shap_display)[:, :, :3]
    orig_float   = original.astype(np.float32) / 255.0
    shap_overlay = (0.55 * orig_float + 0.45 * shap_colored).clip(0, 1)

    axes[i, 0].imshow(original)
    axes[i, 0].set_ylabel(
        f"{res['sample']['plant']}\n{disease}",
        fontsize=7, rotation=0, labelpad=60, va="center"
    )

    im = axes[i, 1].imshow(shap_display, cmap="RdBu_r", vmin=0, vmax=1)
    axes[i, 1].set_xlabel(f"conf={conf:.3f}", fontsize=7)

    axes[i, 2].imshow(shap_overlay)

    for j in range(3):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.suptitle(
    "SHAP Pixel Attributions — ExplainPlan-Vision Phase 2",
    fontsize=12, fontweight="bold", y=1.005
)
plt.tight_layout()
plt.savefig(f"{CONFIG['xai_output_dir']}/shap/shap_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("SHAP grid saved")

---
## Cell 15 — LIME engine

### Background

LIME (Local Interpretable Model-agnostic Explanations) works by perturbing the input image — randomly masking superpixel segments — and training a simple linear model on the resulting predictions. The linear model's coefficients tell us which superpixels contributed positively or negatively to the prediction.

LIME is model-agnostic: it treats the neural network as a black box and probes it through its input-output behaviour. This makes it a useful cross-check against Grad-CAM, which requires access to internal gradients. When Grad-CAM and LIME agree on the important regions, the explanation is more trustworthy.

### Why LIME complements Grad-CAM and SHAP

Grad-CAM shows where the network concentrated its attention in the final convolutional feature map. SHAP shows which pixel regions contributed most to the class score averaged over a background distribution. LIME shows which superpixel-level regions the model relies on locally, independently of its internal architecture. A region that appears in all three explanations is a high-confidence explanation.

In [ ]:
class LIMEEngine:
    """
    LIME image explainer wrapping the plant disease model.

    LIME requires a prediction function that accepts a batch of uint8
    numpy images (N, H, W, 3) and returns a probability matrix (N, C).
    """

    def __init__(self, model, device, num_samples=1000):
        self.model       = model
        self.device      = device
        self.num_samples = num_samples
        self.explainer   = lime_image.LimeImageExplainer()

    def _batch_predict(self, images):
        """
        Prediction function for LIME.
        Accepts (N, H, W, 3) uint8 numpy array, returns (N, C) float32.
        """
        self.model.eval()
        batch_tensors = []
        for img in images:
            t = inference_transform(image=img)["image"]
            batch_tensors.append(t)

        batch = torch.stack(batch_tensors).to(self.device)

        with torch.no_grad():
            logits, _ = self.model(batch)
            probs     = torch.softmax(logits.float(), dim=1)

        return probs.cpu().numpy()

    def explain(self, image_rgb, top_class_idx, num_features=10):
        """
        Compute LIME explanation for the predicted class.

        Parameters
        ----------
        image_rgb      : uint8 numpy array (H, W, 3)
        top_class_idx  : int — the class to explain
        num_features   : int — number of superpixels to highlight

        Returns
        -------
        explanation    : LIME Explanation object
        lime_mask      : (H, W, 3) uint8 overlay image
        """
        # LIME expects the image at its natural resolution; it handles
        # resizing internally via the prediction function wrapper.
        explanation = self.explainer.explain_instance(
            image_rgb,
            self._batch_predict,
            top_labels=3,
            hide_color=0,
            num_samples=self.num_samples
        )

        temp, mask = explanation.get_image_and_mask(
            top_class_idx,
            positive_only=True,
            num_features=num_features,
            hide_rest=False
        )

        lime_overlay = mark_boundaries(temp / 255.0, mask)
        lime_overlay = (lime_overlay * 255).astype(np.uint8)

        return explanation, lime_overlay, mask


lime_engine = LIMEEngine(
    model,
    CONFIG["device"],
    num_samples=CONFIG["lime_num_samples"]
)
print("LIMEEngine ready")

---
## Cell 16 — LIME explanations for demo images

LIME is the slowest of the three methods because it runs hundreds of forward passes per image. On a T4 GPU with `num_samples=1000`, each image takes roughly 20-40 seconds. We run it only on the demo set.

In [ ]:
lime_results = []

for res in tqdm(gradcam_results, desc="LIME"):
    image_rgb  = res["image_rgb"]
    prediction = res["prediction"]
    top_idx    = class_to_idx[prediction["disease"]]

    explanation, lime_overlay, mask = lime_engine.explain(
        image_rgb, top_idx, num_features=CONFIG["lime_num_features"]
    )

    label_safe = res["sample"]["label"].replace("/", "_")
    cv2.imwrite(
        f"{CONFIG['xai_output_dir']}/lime/{label_safe}_lime.png",
        cv2.cvtColor(lime_overlay, cv2.COLOR_RGB2BGR)
    )

    lime_results.append({
        "sample"       : res["sample"],
        "prediction"   : prediction,
        "image_rgb"    : image_rgb,
        "lime_overlay" : lime_overlay,
        "mask"         : mask,
    })

print(f"LIME explanations computed for {len(lime_results)} images")

---
## Cell 17 — Three-method comparison figure

This is the primary research figure of Phase 2. Each row corresponds to one disease class. The four columns are: original image, Grad-CAM++ overlay, SHAP attribution, and LIME superpixel mask. A reviewer or reader can see at a glance whether the three methods agree on the important regions.

Convergence across methods strengthens the claim that the model is attending to genuine disease symptoms rather than confounding artefacts (background, label stickers, lighting).

In [ ]:
n   = len(gradcam_results)
fig, axes = plt.subplots(n, 4, figsize=(17, n * 3.2))

col_titles = ["Original", "Grad-CAM++", "SHAP", "LIME"]
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=11, fontweight="bold", pad=10)

for i in range(n):
    gc_res   = gradcam_results[i]
    sh_res   = shap_results[i]
    li_res   = lime_results[i]

    original  = cv2.resize(gc_res["image_rgb"], (224, 224))
    pred      = gc_res["prediction"]
    disease   = pred["disease_type"]
    conf      = pred["confidence"]
    plant     = gc_res["sample"]["plant"]

    # Shap overlay
    shap_colored = cm.RdBu_r(sh_res["shap_display"])[:, :, :3]
    orig_float   = original.astype(np.float32) / 255.0
    shap_overlay = (0.55 * orig_float + 0.45 * shap_colored).clip(0, 1)

    axes[i, 0].imshow(original)
    axes[i, 0].set_ylabel(
        f"{plant}\n{disease[:20]}\nconf={conf:.3f}",
        fontsize=7.5, rotation=0, labelpad=75, va="center"
    )

    axes[i, 1].imshow(gc_res["overlay"])
    axes[i, 2].imshow(shap_overlay)
    axes[i, 3].imshow(li_res["lime_overlay"])

    for j in range(4):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.suptitle(
    "Three-Method XAI Comparison — ExplainPlan-Vision Phase 2\n"
    "Left to right: original / Grad-CAM++ / SHAP / LIME",
    fontsize=12, fontweight="bold", y=1.005
)
plt.tight_layout()
plt.savefig(
    f"{CONFIG['xai_output_dir']}/comparison/three_method_comparison.png",
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Three-method comparison figure saved")

---
## Cell 18 — Textual explanation generator

Visual heatmaps are valuable for researchers, but a farmer in the field needs language. This module generates four types of natural language explanations for any prediction. These are rule-based templates grounded in the model's output statistics — not LLM-generated text.

In Phase 5, these templates will be replaced with user-adaptive generation that selects vocabulary and technical depth based on the identified user type.

In [ ]:
# Disease-specific knowledge base for textual explanations.
# This will grow substantially in Phase 3 when the planning engine is added.
DISEASE_KNOWLEDGE = {
    "Early blight": {
        "pathogen"    : "fungal (Alternaria solani)",
        "visual_cues" : "dark concentric rings on older leaves",
        "spread"      : "spore dispersal via wind and water splash",
    },
    "Late blight": {
        "pathogen"    : "oomycete (Phytophthora infestans)",
        "visual_cues" : "water-soaked lesions with pale green border",
        "spread"      : "rapid in cool, wet conditions",
    },
    "Bacterial spot": {
        "pathogen"    : "bacterial (Xanthomonas spp.)",
        "visual_cues" : "small water-soaked spots with yellow halo",
        "spread"      : "rain splash and infected transplants",
    },
    "Septoria leaf spot": {
        "pathogen"    : "fungal (Septoria lycopersici)",
        "visual_cues" : "small circular spots with dark borders and grey centres",
        "spread"      : "splashing water and infected plant debris",
    },
    "Leaf Mold": {
        "pathogen"    : "fungal (Passalora fulva)",
        "visual_cues" : "pale greenish-yellow spots above, olive-green mould below",
        "spread"      : "high humidity and poor air circulation",
    },
    "Target Spot": {
        "pathogen"    : "fungal (Corynespora cassiicola)",
        "visual_cues" : "brown spots with concentric rings resembling a target",
        "spread"      : "warm wet conditions",
    },
    "Spider mites Two spotted spider mite": {
        "pathogen"    : "arthropod pest (Tetranychus urticae)",
        "visual_cues" : "stippling, bronzing, and fine webbing on leaves",
        "spread"      : "wind, contact, and contaminated equipment",
    },
    "Tomato YellowLeaf  Curl Virus": {
        "pathogen"    : "viral (TYLCV)",
        "visual_cues" : "upward leaf curl and yellowing of margins",
        "spread"      : "whitefly (Bemisia tabaci) vector",
    },
    "Tomato mosaic virus": {
        "pathogen"    : "viral (ToMV)",
        "visual_cues" : "mosaic discolouration and leaf distortion",
        "spread"      : "mechanical contact and contaminated tools",
    },
}


def generate_explanation(prediction, gradcam_heatmap):
    """
    Generate four types of natural language explanation for a prediction.

    Returns a dictionary with:
        why          - why this class was predicted
        why_not      - why the runner-up was rejected
        confidence   - interpretation of the confidence score
        counterfactual - what would change the prediction
    """
    disease       = prediction["disease_type"]
    confidence    = prediction["confidence"]
    severity      = prediction["severity"]
    top3          = prediction["top3_predictions"]
    runner_up     = top3[1]["class"] if len(top3) > 1 else None
    runner_up_conf= top3[1]["confidence"] if len(top3) > 1 else 0.0
    conf_gap      = round(confidence - runner_up_conf, 4)
    is_healthy    = prediction["is_healthy"]

    # Determine how much of the heatmap is concentrated
    # in the top-20% of activation values (focus score)
    threshold    = np.percentile(gradcam_heatmap, 80)
    focus_ratio  = float((gradcam_heatmap >= threshold).mean())
    focus_desc   = "localised" if focus_ratio < 0.08 else "distributed"

    # Retrieve known disease characteristics
    knowledge = DISEASE_KNOWLEDGE.get(disease, {})
    visual_cue = knowledge.get("visual_cues", "symptom patterns on the leaf surface")
    pathogen   = knowledge.get("pathogen", "an identified pathogen")

    if is_healthy:
        why_text = (
            f"The leaf appears healthy. The model found no visual patterns "
            f"consistent with any of the {NUM_CLASSES - 1} disease classes "
            f"in its training distribution. Confidence: {confidence:.1%}."
        )
    else:
        why_text = (
            f"The model predicted {disease} ({pathogen}) with {confidence:.1%} confidence. "
            f"The decision was driven by {focus_desc} activation regions consistent "
            f"with {visual_cue}. The severity is assessed as {severity} based on "
            f"the prediction confidence."
        )

    # Why-not explanation
    if runner_up and not is_healthy:
        ru_disease = runner_up.split("___")[-1].replace("_", " ") if "___" in runner_up \
                     else " ".join(runner_up.split("_")[1:])
        ru_knowledge = DISEASE_KNOWLEDGE.get(ru_disease, {})
        ru_cue       = ru_knowledge.get("visual_cues", "different symptom patterns")
        why_not_text = (
            f"{ru_disease} was considered but rejected. Its characteristic features "
            f"({ru_cue}) were not prominently present in this image. "
            f"The confidence gap between the two predictions was {conf_gap:.1%}."
        )
    else:
        why_not_text = "No disease classes had significant competing activation."

    # Confidence interpretation
    if confidence >= 0.90:
        conf_text = (
            f"Confidence is high ({confidence:.1%}). The visual evidence in this "
            f"image strongly matches the learned pattern for {disease}."
        )
    elif confidence >= 0.70:
        conf_text = (
            f"Confidence is moderate ({confidence:.1%}). The system recommends "
            f"verifying this diagnosis with a second image or expert consultation."
        )
    else:
        conf_text = (
            f"Confidence is low ({confidence:.1%}). Treat this prediction with "
            f"caution. Image quality or unusual symptom presentation may be a factor."
        )

    # Counterfactual
    counterfactual_text = (
        f"If the {focus_desc} lesion regions were absent or less pronounced, "
        f"the prediction would likely shift toward "
        f"{runner_up.split('___')[-1].replace('_', ' ') if runner_up and '___' in runner_up else 'a related condition'} "
        f"or Healthy. Reducing image noise and ensuring consistent lighting "
        f"would improve prediction stability."
    )

    return {
        "why"            : why_text,
        "why_not"        : why_not_text,
        "confidence"     : conf_text,
        "counterfactual" : counterfactual_text,
    }


print("generate_explanation() ready")

---
## Cell 19 — Textual explanation demo

In [ ]:
for res in gradcam_results[:3]:
    explanation = generate_explanation(res["prediction"], res["heatmap"])
    print("=" * 60)
    print(f"Image  : {res['sample']['plant']} / {res['prediction']['disease_type']}")
    print(f"Conf   : {res['prediction']['confidence']:.3f}  Severity: {res['prediction']['severity']}")
    print()
    print("WHY")
    print(explanation["why"])
    print()
    print("WHY NOT")
    print(explanation["why_not"])
    print()
    print("CONFIDENCE")
    print(explanation["confidence"])
    print()
    print("COUNTERFACTUAL")
    print(explanation["counterfactual"])
    print()

---
## Cell 20 — Unified explain() function

This is the primary interface that Phase 3 will call. It accepts a raw image path and returns the complete enriched prediction dictionary including all three visual explanations and all four textual explanations. Phase 3's planning engine reads the `prediction` and `severity` fields to generate treatment plans. Phase 5's user-adaptive module reads the `explanation` fields to select the right vocabulary.

In [ ]:
def explain(image_path, model, config, run_lime=True):
    """
    Full explainability pipeline for a single image.

    Parameters
    ----------
    image_path : str — path to input image
    model      : PlantDiseaseModel
    config     : CONFIG dict
    run_lime   : bool — LIME is slow; set False for real-time use

    Returns
    -------
    A dictionary with:
        prediction       - full Phase 1 prediction dict
        gradcam_heatmap  - (H, W) float32 array
        gradcam_overlay  - (H, W, 3) uint8 RGB
        shap_display     - (H, W) float32 attribution map
        lime_overlay     - (H, W, 3) uint8 RGB (or None if run_lime=False)
        explanation      - dict with why/why_not/confidence/counterfactual
        image_rgb        - original image
    """
    image_rgb  = load_image_rgb(image_path)
    prediction = predict(image_rgb, model, config)
    top_idx    = class_to_idx[prediction["disease"]]

    # Grad-CAM
    heatmap, gc_overlay = gradcam_engine.generate(image_rgb)

    # SHAP
    _, shap_display = shap_engine.explain(image_rgb, top_idx)

    # LIME (optional)
    lime_overlay = None
    if run_lime:
        _, lime_overlay, _ = lime_engine.explain(
            image_rgb, top_idx, num_features=config["lime_num_features"]
        )

    # Textual explanations
    text_explanation = generate_explanation(prediction, heatmap)

    return {
        "prediction"      : prediction,
        "gradcam_heatmap" : heatmap,
        "gradcam_overlay" : gc_overlay,
        "shap_display"    : shap_display,
        "lime_overlay"    : lime_overlay,
        "explanation"     : text_explanation,
        "image_rgb"       : image_rgb,
    }


print("explain() unified interface ready")
print()
print("Output schema for Phase 3:")
print("  explain(path) -> {")
print("    prediction: {disease, confidence, severity, feature_embedding, ...}")
print("    gradcam_overlay: (224, 224, 3) uint8")
print("    shap_display:    (224, 224) float32")
print("    lime_overlay:    (H, W, 3) uint8")
print("    explanation: {why, why_not, confidence, counterfactual}")
print("  }")

---
## Cell 21 — Embedding-based counterfactual retrieval

A counterfactual example is a real image from the dataset that is closest to the query image in feature space but belongs to a different class. Showing the farmer a real photograph of the alternative disease provides more grounded evidence than any text description could.

We compute cosine similarity between the query embedding and all test-set embeddings, then retrieve the nearest example from each of the top-2 alternative classes.

In [ ]:
# Build an embedding index over a representative sample of the full dataset.
# We sample 30 images per class to keep memory manageable.

INDEX_SAMPLE_PER_CLASS = 30
index_records = []

for cls in tqdm(classes, desc="Building embedding index"):
    cls_dir = dataset_path / cls
    imgs    = [str(f) for f in cls_dir.iterdir()
               if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    chosen  = random.sample(imgs, min(INDEX_SAMPLE_PER_CLASS, len(imgs)))

    model.eval()
    with torch.no_grad():
        for path in chosen:
            img    = load_image_rgb(path)
            tensor = preprocess_for_model(img)
            feat   = model.extract_features(tensor)[0].cpu().float().numpy()
            index_records.append({"path": path, "label": cls, "embedding": feat})

index_df         = pd.DataFrame(index_records)
embedding_matrix = np.stack(index_df["embedding"].values)  # (N, 1280)

print(f"Embedding index built: {len(index_df)} images")


def retrieve_counterfactual(query_embedding, query_class, top_k=1):
    """
    Find the most similar image from a different class.

    Returns a list of {path, label, similarity} dicts.
    """
    sims   = cosine_similarity(query_embedding.reshape(1, -1), embedding_matrix)[0]
    # Mask same class
    is_other = index_df["label"].values != query_class
    sims_filtered = np.where(is_other, sims, -1)
    top_indices   = np.argsort(sims_filtered)[::-1][:top_k]

    return [
        {
            "path"       : index_df.iloc[idx]["path"],
            "label"      : index_df.iloc[idx]["label"],
            "similarity" : round(float(sims[idx]), 4)
        }
        for idx in top_indices
    ]


print("retrieve_counterfactual() ready")

---
## Cell 22 — Counterfactual retrieval demo

In [ ]:
n   = min(4, len(gradcam_results))
fig, axes = plt.subplots(n, 2, figsize=(9, n * 3.5))

axes[0, 0].set_title("Query image",         fontsize=10, fontweight="bold")
axes[0, 1].set_title("Counterfactual (nearest different class)", fontsize=10, fontweight="bold")

for i in range(n):
    res      = gradcam_results[i]
    pred     = res["prediction"]
    emb      = pred["feature_embedding"]
    cf_list  = retrieve_counterfactual(emb, pred["disease"], top_k=1)

    query_img = cv2.resize(res["image_rgb"], (224, 224))
    cf_img    = load_image_rgb(cf_list[0]["path"])
    cf_img    = cv2.resize(cf_img, (224, 224))

    cf_label  = cf_list[0]["label"].split("___")[-1].replace("_", " ") \
                if "___" in cf_list[0]["label"] \
                else " ".join(cf_list[0]["label"].split("_")[1:])

    axes[i, 0].imshow(query_img)
    axes[i, 0].set_ylabel(
        f"{pred['plant']}\n{pred['disease_type']}",
        fontsize=8, rotation=0, labelpad=75, va="center"
    )
    axes[i, 0].set_xlabel(f"conf={pred['confidence']:.3f}", fontsize=8)

    axes[i, 1].imshow(cf_img)
    axes[i, 1].set_xlabel(f"{cf_label[:30]}  (sim={cf_list[0]['similarity']:.3f})", fontsize=8)

    for j in range(2):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.suptitle(
    "Counterfactual Retrieval — Most Similar Image from Different Class",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig(f"{CONFIG['xai_output_dir']}/counterfactual/counterfactual_pairs.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Counterfactual figure saved")

---
## Cell 23 — XAI method agreement analysis

This cell computes a quantitative measure of how much Grad-CAM and SHAP agree with each other per image. We use Spearman correlation between the flattened heatmap and the SHAP attribution map. High correlation means both methods point to the same pixels, strengthening the explanation's credibility.

This analysis produces a table that can be included directly in the paper's explainability evaluation section.

In [ ]:
from scipy.stats import spearmanr

agreement_rows = []

for gc_res, sh_res in zip(gradcam_results, shap_results):
    heatmap      = gc_res["heatmap"].flatten()
    shap_flat    = sh_res["shap_display"].flatten()
    corr, pvalue = spearmanr(heatmap, shap_flat)

    agreement_rows.append({
        "disease"              : gc_res["prediction"]["disease_type"],
        "confidence"           : gc_res["prediction"]["confidence"],
        "spearman_gradcam_shap": round(corr, 4),
        "p_value"              : round(pvalue, 6),
        "significant"          : pvalue < 0.05
    })

agreement_df = pd.DataFrame(agreement_rows)
print("XAI Method Agreement (Grad-CAM vs SHAP, Spearman rank correlation)")
print(agreement_df.to_string(index=False))
print()
print(f"Mean Spearman r : {agreement_df['spearman_gradcam_shap'].mean():.4f}")
agreement_df.to_csv(f"{CONFIG['xai_output_dir']}/comparison/method_agreement.csv", index=False)

---
## Cell 24 — Save Phase 2 outputs for Phase 3

We save the explainability outputs in a format that Phase 3's planning engine can read. The most important file is `explain_schema.json`, which documents the structure of the `explain()` function's output — the interface contract between Phase 2 and Phase 3.

In [ ]:
# Document the explain() output schema for Phase 3
explain_schema = {
    "description": "Output schema of the explain() function in Phase 2",
    "fields": {
        "prediction": {
            "disease"          : "full class label string",
            "plant"            : "plant type parsed from label",
            "disease_type"     : "disease name parsed from label",
            "confidence"       : "float in [0,1]",
            "severity"         : "high | medium | low",
            "top3_predictions" : "list of {class, confidence}",
            "feature_embedding": "numpy array shape (1280,)",
            "is_healthy"       : "boolean"
        },
        "gradcam_heatmap"  : "numpy float32 (224, 224) in [0,1]",
        "gradcam_overlay"  : "numpy uint8 (224, 224, 3)",
        "shap_display"     : "numpy float32 (224, 224) in [0,1]",
        "lime_overlay"     : "numpy uint8 (H, W, 3) or None",
        "explanation": {
            "why"            : "string — why this class was predicted",
            "why_not"        : "string — why runner-up was rejected",
            "confidence"     : "string — confidence interpretation",
            "counterfactual" : "string — what would change the prediction"
        }
    },
    "phase3_fields_used": ["prediction.disease", "prediction.severity",
                           "prediction.confidence", "prediction.is_healthy",
                           "prediction.plant", "explanation.why",
                           "explanation.why_not"]
}

with open(f"{CONFIG['xai_output_dir']}/explain_schema.json", "w") as f:
    json.dump(explain_schema, f, indent=2)

# Save agreement table
agreement_df.to_csv(
    f"{CONFIG['xai_output_dir']}/comparison/method_agreement.csv", index=False
)

print("Phase 2 outputs saved")
print()
for root, dirs, files in os.walk("outputs/xai"):
    level   = root.replace("outputs/xai", "").count(os.sep)
    indent  = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"  {indent}{file}")

---
## Cell 25 — Phase 2 summary and bridge to Phase 3

### Results

| Component | Method | Output |
|-----------|--------|--------|
| Spatial explanation | Grad-CAM++ | Heatmap showing which leaf regions activated |
| Feature attribution | SHAP DeepExplainer | Per-pixel attribution relative to background |
| Decision boundary | LIME | Superpixel mask of locally important regions |
| Textual reasoning | Rule-based templates | why / why-not / confidence / counterfactual |
| Counterfactual retrieval | Cosine embedding search | Nearest real image from different class |
| Method agreement | Spearman correlation | Quantitative Grad-CAM vs SHAP agreement |

### What Phase 3 receives from Phase 2

The `explain()` function returns a fully enriched prediction dictionary. Phase 3 reads the following fields to drive treatment planning:

```
explain(image_path) ->
    prediction.disease      → knowledge base lookup
    prediction.severity     → treatment urgency determination
    prediction.confidence   → plan confidence gating
    prediction.plant        → crop-specific rule selection
    prediction.is_healthy   → early exit (no treatment needed)
    explanation.why         → include in plan rationale
    explanation.why_not     → include in alternative plan
```

### Phase 3 preview — Sequential Planning Engine

Phase 3 will implement a rule-based planner that takes the `explain()` output and produces a sequenced treatment plan:

```
{
    "steps": [
        {"order": 1, "action": "Apply copper-based fungicide",
         "rationale": "Fungal infection confirmed with 91% confidence"},
        {"order": 2, "action": "Remove and destroy infected leaves",
         "rationale": "Localised activation suggests early-stage infection"},
        {"order": 3, "action": "Reduce overhead irrigation",
         "rationale": "Reduces spore dispersal via water splash"},
        {"order": 4, "action": "Monitor for 5 days",
         "rationale": "Re-evaluate if lesion spread exceeds 20% leaf area"}
    ],
    "urgency": "high",
    "explanation_source": "Phase 2 Grad-CAM + textual engine"
}
```

---

**GitHub note:** Commit this notebook as `notebooks/phase2_explainability_engine.ipynb`. Save the `outputs/xai/` directory as a new Kaggle dataset named `explainplan-phase2-outputs` and add it as input to the Phase 3 notebook alongside the Phase 1 outputs dataset.